# NotebookLM Kit — Video Pipeline

Generates one video per source, downloads all as MP4.

1. **Setup** — authenticate and verify `tsx`
2. **Config** — set your notebook ID
3. **Template** — choose format, visual style, and optional focus prompt
4. **List Sources** — preview sources; pick indices to filter
5. **Create** — submit one video job per source; jobs saved to disk
6. **Poll** — wait for generation, or resume from a previous run
7. **Download** — save each video as `<timestamp>_<Notebook>__<Source>.mp4`


In [ ]:
import importlib, pipeline, pipeline._core, pipeline._sources, pipeline._artifacts, pipeline.config
importlib.reload(pipeline.config)
importlib.reload(pipeline._core)
importlib.reload(pipeline._sources)
importlib.reload(pipeline._artifacts)
importlib.reload(pipeline)

from pipeline import load_credentials, check_tsx, login, SDK_ROOT

# First time only — opens a browser window for Google login, saves credentials.json:
# login()

creds = load_credentials(mode="patchright")
check_tsx()

Credentials ready — token: 42 chars, cookies: 2548 chars
tsx: tsx v4.22.3
node v20.17.0


## Config

Find your notebook ID in the NotebookLM URL:  
`https://notebooklm.google.com/notebook/` **`<YOUR_NOTEBOOK_ID>`**

In [ ]:
NOTEBOOK_ID = "0a79eb08-bbdb-48a0-9b23-5a2ca44e368f"  # ← paste your notebook ID here

## List Sources

`list_sources(notebook_id, creds)` — prints an indexed table of all sources (title, status, full ID).  
Filter by index before passing to **Create**:  `sources[:1]` to test one first  /  `[sources[i] for i in [0, 2]]`


In [ ]:
from pipeline import list_sources

sources = list_sources(NOTEBOOK_ID, creds)



Notebook :  4 Napoleon
+---+----------------------+------------+--------------------------------------+
| # | Title                | Status     | Source ID                            |
+---+----------------------+------------+--------------------------------------+
| 0 | 00. Introduction.txt | READY      | 89fd686f-2cd5-4ef0-b427-50424c8d24b4 |
| 1 | 01. Corsica.txt      | READY      | ce3a771d-70b1-44ee-949c-8e83e4a21d52 |
| 2 | 02. Revolution.txt   | READY      | ebd71561-b799-4de4-bde9-1086cef8ae7e |
| 3 | 03. Desire.txt       | READY      | 5f2ccf0c-095a-499a-889f-9abb1bc92525 |
| 4 | 04. Italy.txt        | READY      | 56a7c070-e2d7-41f0-8467-5fe755e38f42 |
| 5 | 05. Victory.txt      | READY      | 36ac048c-22b5-403d-be4e-8d80f1cc68fc |
| 6 | 06. Peace.txt        | READY      | 3677333b-9223-4605-8920-f0b914ab52a5 |
| 7 | 07. Egypt.txt        | READY      | 8606ebfd-a9fb-443e-9779-cda83dac7c94 |
| 8 | 08. Acre.txt         | READY      | 5352bfee-94aa-4f22-bff3-7f9984c43aa6 |
| 9 

## Video Template

Set format, visual style, and an optional focus prompt.


In [ ]:
from config import VIDEO_PROMPT

VIDEO_INSTRUCTIONS = VIDEO_PROMPT.strip()  # optional focus / style prompt — leave empty for default

# FORMAT:       1 = Explainer   2 = Brief      3 = Cinematic  (Cinematic ignores visualStyle)
# VISUAL STYLE: 1 = Auto        2 = Classic    3 = Whiteboard   4 = Heritage  9 = Kawaii
VIDEO_CUSTOMIZATION = {
    "format":      1,
    "visualStyle": 4,
    "focus":       VIDEO_INSTRUCTIONS,
    # "language": "en",  # BCP-47 — omit to use notebook default
}


## Create Videos

`create_artifacts(notebook_id, type, sources, customization, instructions, creds, *, dry_run=False)`

- Jobs saved to `jobs/video/<timestamp>_<NotebookName>.json`
- `instructions` — additional prompt; pass `None` to omit
- `dry_run=True` — preview which sources would be submitted without calling the API
- Skip this cell and run **Resume** below if jobs were already submitted


In [ ]:
from pipeline import create_artifacts

video_jobs = create_artifacts(
    NOTEBOOK_ID, "VIDEO", sources[14:],
    VIDEO_CUSTOMIZATION, VIDEO_INSTRUCTIONS or None,
    creds,
    dry_run=False,  # ← set to False (or remove) to actually submit
)

# sources = sources[:1]                   # test with one first
# sources = [sources[i] for i in [0, 2]]  # specific picks



Submitted 18 job(s)  →  jobs\video\20260531_163358 4 Napoleon.json
+---+--------------------+----------------------------------------------+
| # | Source             | Artifact ID                                  |
+---+--------------------+----------------------------------------------+
| 0 | 14. Amiens.txt     | 7abf175f-b39a-4ad7-aedf-2fecdeccb8b0         |
| 1 | 15. Coronation.txt | ae6f8f5f-0bdb-4bcd-a7e8-086de60a8041         |
| 2 | 16. Austerlitz.txt | a3890330-3f87-4f8c-9f12-46d6bd60262f         |
| 3 | 17. Jena.txt       | f56f6788-7e5f-42d1-ad0f-3b657b9f9adc         |
| 4 | 18. Blockades.txt  | 6a896966-471f-480d-ba06-4e8d566772bc         |
| 5 | 19. Tilsit.txt     | 7093de31-4221-4d70-a00a-5b88b23ff6e1         |
| 6 | 20. Iberia.txt     | 991b0c2a-c319-45ff-9ec6-ff9c006515c4         |
| 7 | 21. Wagram.txt     | ed562c7f-f87c-40b2-b8e0-aad486544fac         |
| 8 | 22. Zenith.txt     | c792a615-ecf1-4709-9c41-7eb35bbe578b         |
| 9 | 23. Russia.txt     | 1d66f697-af3a-4f7

## Poll Until Ready

`poll_jobs(jobs, notebook_id, creds, *, interval=60, max_wait_min=30)`

- `interval` — seconds between checks (videos take longer; 60s recommended)
- `max_wait_min` — timeout; returns `False` if exceeded (safe to re-run)

**Resume** (cell below): run instead of **Create** when continuing a previous session.


In [ ]:
# ── Resume cell — run this instead of Create if you already submitted jobs ───
from pipeline import load_jobs
video_jobs = load_jobs("VIDEO")


Loaded 1 job(s) from 20260530_090711_Mastering_Modern_Data_Orchestr.json
+---+--------------------------------------------------+----------------------------------------------+
| # | Source                                           | Artifact ID                                  |
+---+--------------------------------------------------+----------------------------------------------+
| 0 | Best Practices for Machine Learning with Dagster | e5302356-b2c7-4e6a-b8de-096242f3bb91         |
+---+--------------------------------------------------+----------------------------------------------+


In [ ]:
from pipeline import poll_jobs

poll_jobs(video_jobs, NOTEBOOK_ID, creds, interval=60, max_wait_min=60)

  ✓ Best Practices for Machine Learning with Dagster        READY

✅ All artifacts ready — proceed to download.


True

## Download

Each video is saved as `<source title>.mp4` in `outputs/video/`.

In [ ]:
from pipeline import download_artifacts

download_artifacts(video_jobs, NOTEBOOK_ID, "VIDEO", creds)

  Resolving video URL for: Best Practices for Machine Learning with Dagster …

Downloaded 0 / 1 artifact(s)  →  d:\Core\_Code D\notebooklm-kit\outputs\video
+---+--------+----------------------------------------------+------------+
| # | Source | File                                         | Status     |
+---+--------+----------------------------------------------+------------+
+---+--------+----------------------------------------------+------------+
  x  Best Practices for Machine Learning with Dagster: asyncio.run() cannot be called from a running event loop
+---+--------+----------------------------------------------+------------+


d:\Core\_Code D\notebooklm-kit\pipeline\_artifacts.py:443: RuntimeWarning: coroutine '_resolve_and_download_video' was never awaited


[]